## Setup

### General

In [22]:
import pandas as pd
import numpy as np
import os
import sys

### Paths & Logger

In [23]:
PATH_ORG_DATA = "data/compas-scores-two-years.csv"
PATH_SYNTHETIC_DATA = "data/synthetic/"
PATH_FOLDER_MODELS = "models/"
PATH_RUN_FOLDER = "_run/"

if not os.path.exists(PATH_FOLDER_MODELS):
    os.makedirs(PATH_FOLDER_MODELS)
if not os.path.exists(PATH_SYNTHETIC_DATA):
    os.makedirs(PATH_SYNTHETIC_DATA)

In [24]:
import logging
rootLogger = logging.getLogger()

fileHandler = logging.FileHandler(
    "{0}/{1}.log".format(PATH_RUN_FOLDER, f"log-{pd.Timestamp.now().strftime('%Y-%m-%d_%H-%M-%S')}"))
fileFormatter = logging.Formatter(
    "%(asctime)s [%(funcName)s] [%(levelname)s]  %(message)s")
fileHandler.setFormatter(fileFormatter)
rootLogger.addHandler(fileHandler)

consoleHandler = logging.StreamHandler()
consoleFormatter = logging.Formatter("[%(levelname)s]: %(message)s")
consoleHandler.setFormatter(consoleFormatter)
rootLogger.addHandler(consoleHandler)

### Dataset

Sensitive attributes include:
- sex
- race

Target variable:
- is_recid

In [25]:
scores_two_years = pd.read_csv(PATH_ORG_DATA)

columns_to_include = [
    "name",
    "first",
    "last",
    "sex",
    "dob",
    "age",
    "age_cat",
    "race",
    "juv_fel_count",
    "juv_misd_count",
    "juv_other_count",
    "priors_count",
    "c_case_number",
    "c_charge_degree",
    "c_charge_desc",
    "r_case_number",
    "r_charge_degree",
    "r_charge_desc",
    "in_custody",
    "out_custody",
    "two_year_recid",
    "is_recid",
]

dataset = scores_two_years[columns_to_include].copy()

n = dataset.shape[0]

In [26]:
categorical_columns = [
    "name",
    "first",
    "last",
    "sex",
    "dob",
    "age_cat",
    "race",
    "c_case_number",
    "c_charge_degree",
    "c_charge_desc",
    "r_case_number",
    "r_charge_degree",
    "r_charge_desc",
    "in_custody",
    "out_custody",
    "two_year_recid",
    "is_recid",
]

## Data Preprocessing

When `is_recid` is 0 there should not be any values in `r_charge_degree` and `r_charge_desc`, as they contain information about the crime they were re-arested for. Thus, we made a separate category "Not applicable" to make it clear which values are truly missing.

In [27]:
recid_charge_cols = [
    "r_case_number",
    "r_charge_degree",
    "r_charge_desc",
]

for col in recid_charge_cols:
    dataset[col] = np.where(
        (dataset["is_recid"] == 0), "Not applicable", dataset[col]
    )

The `race` attribute is distributed as indicated in the table below. The ammount of data on "Hispanic", "Asian", and "Native American" individually is not enough to be significant, therefore we combine these together with "Other".

| Race             | Count | Percent |
| --- | --- | ---  |
| African-American |  3696 |  51.23% |
| Caucasian        |  2454 |  34.02% |
| Hispanic         |   637 |   8.83% |
| Other            |   377 |   5.23% |
| Asian            |    32 |   0.44% |
| Native American  |    18 |   0.25% |

In [28]:
other_races = ["Hispanic", "Asian", "Native American"]

dataset["race"] = dataset["race"].replace(other_races, "Other")

In [29]:
dataset.to_csv("data/preprocessed_real_data.csv", index=False)

## Model Training

#### Model Setup

In [30]:
# To make sure that FairGAN can be imported
path_to_dir = os.environ['VIRTUAL_ENV'][:-5]
sys.path.append(path_to_dir)

In [31]:
from ctgan import CTGAN
from ctgan import TVAE
from tabfairgan import TFG
from fairgan import FairGAN

import pickle

In [32]:
def save_successful(model, model_name: str, synthetic_data: pd.DataFrame):
    try:
        synthetic_data.to_csv(
            PATH_SYNTHETIC_DATA + f"{model_name}_synthetic.csv", index=False
        )

        with open(PATH_FOLDER_MODELS + f"{model_name}_model.pkl", "wb") as f:
            pickle.dump(model, f)
    except Exception as e:
        rootLogger.error(e)

#### CTGAN

The CTGAN model learns the joint distribution of the dataset.

In [33]:
def run_CTGAN(df: pd.DataFrame, num_epochs: int, batch_size: int, verbose: bool, synthetic_data_count: int):
    rootLogger.info("\n\nStarting to train CTGAN model")

    try:
        # Train CTGAN model
        ctgan_df = df.copy()

        ctgan_model = CTGAN(epochs=num_epochs,
                            batch_size=batch_size,
                            verbose=verbose)
        ctgan_model.fit(ctgan_df, discrete_columns=categorical_columns)

    # Generate synthetic data using CTGAN
        ctgan_synthetic_data = ctgan_model.sample(synthetic_data_count)

        save_successful(ctgan_model, "ctgan", ctgan_synthetic_data)

    except Exception as e:
        rootLogger.error(e)

#### TVAE

TVAE learns a latent representation of the data and reconstructs it through a decoder network.

Unlike CTGAN, TVAE optimizes a probabilistic objective rather than using adversarial training.

In [34]:
def run_TVAE(df: pd.DataFrame, num_epochs: int, batch_size: int, verbose: bool, synthetic_data_count: int):
    rootLogger.info("\n\nStarting to train TVAE model")

    try:
        # Train TVAE model
        tvae_df = df.copy()

        tvae_model = TVAE(epochs=num_epochs,
                          batch_size=batch_size,
                          verbose=verbose)
        tvae_model.fit(tvae_df, discrete_columns=categorical_columns)

    # Generate synthetic data using TVAE
        tvae_synthetic_data = tvae_model.sample(synthetic_data_count)

        save_successful(tvae_model, "tvae", tvae_synthetic_data)
    except Exception as e:
        rootLogger.error(e)
        pass

#### FairGAN

Our implementation of the FairGAN model can be found [here](fairgan.py)

In [35]:
def run_FairGAN(df: pd.DataFrame, num_epochs: int, synthetic_data_count: int, sensitive_feature: str, target_feature: str):
    rootLogger.info("\n\nStarting to train FairGAN with different λ_fair values")

    lambda_values = [0.0, 0.1, 0.5, 0.7, 1.0]

    fairgan_datasets = {}
    fairgan_models = {}

    for lam in lambda_values:
        rootLogger.info(f"\nTraining FairGAN with λ_fair = {lam}")

        fairgan_df = df.copy()

        if lam == 0.0:
            fair_epochs = 0
        else:
            fair_epochs = 50

        try:
            # Initialize model
            fairgan_model = FairGAN(
                df=fairgan_df,
                sensitive_feature=sensitive_feature,
                target_feature=target_feature,
                epochs=num_epochs,
                noise_dim=100,
                lr=0.001,
                verbose=True,
            )

        # Train model
            fairgan_model.train(lambda_fair=lam)

        # Generate synthetic dataset
            synthetic_df = fairgan_model.generate(synthetic_data_count)

            dataset_name = f"FairGAN_lambda_{lam}"
            fairgan_datasets[dataset_name] = synthetic_df
            fairgan_models[dataset_name] = fairgan_model

            save_successful(fairgan_model, dataset_name, synthetic_df)

            rootLogger.info(f"Completed {dataset_name}")

        except Exception as e:
            rootLogger.error(e)

#### TabFairGAN

TabFairGAN differs from standard generative models such as CTGAN in that fairness constraints are incorporated directly into the training process.

Instead of using a separate fit function, the model is trained using a custom training loop.

**Fairness Constraints**

To generate fair synthetic data, you need to pass a dictionary containing the following parameters:

- `fair_epochs`: Number of fair training epochs (integer).
- `lamda`: Lambda parameter controlling the trade-off between fairness and accuracy (float).
- `S`: Protected attribute (string, e.g., "sex").
- `Y`: Decision label (string, e.g., "income").
- `S_under`: Value representing the underprivileged group for the protected attribute (string, e.g., " Female").
- `Y_desire`: Desired value for the label (string, e.g., " >50K").

In [36]:
# TODO: Define proper Y_desire

def run_TabFairGAN(df: pd.DataFrame, num_epochs: int, batch_size: int, synthetic_data_count: int, fairness_config: dict):
    rootLogger.info("\n\nStarting to train TabFairGAN with different λ values")

    lambda_values = [0.0, 0.1, 0.5, 0.7, 1.0]

    tabfairgan_datasets = {}
    tabfairgan_models = {}

    for lam in lambda_values:
        if lam == 0.0:
            fairness_config["fair_epochs"] = 0
        else:
            fairness_config["fair_epochs"] = 50

        fairness_config["lamda"] = lam

        try:
            rootLogger.info(f"\nTraining TabFairGAN with λ = {lam}")

            tfg_df = df.copy()

            # Turn the sensitive and target attribute into categorical string
            tfg_df["sex"] = tfg_df["sex"].astype(str)
            tfg_df["is_recid"] = tfg_df["is_recid"].astype(str)

        # Initialize model
            tfg_model = TFG(
                tfg_df,
                epochs=num_epochs,
                batch_size=batch_size,
                fairness_config=fairness_config,
            )

        # Train model
            tfg_model.train()

        # Generate synthetic data
            tfg_synthetic_data = tfg_model.generate_fake_df(
                num_rows=synthetic_data_count
            )

            dataset_name = f"TabFairGAN_lambda_{lam}"
            tabfairgan_datasets[dataset_name] = tfg_synthetic_data
            tabfairgan_models[dataset_name] = tfg_model

            save_successful(tfg_model, dataset_name, tfg_synthetic_data)

            rootLogger.info(f"Completed {dataset_name}")

        except Exception as e:
            rootLogger.error(e)
            continue

## Model Settings

In [37]:
# Set logging level
rootLogger.setLevel(logging.INFO)

In [38]:
target_feature = "is_recid"
sensitive_feature = "sex"
sensitive_under = "Female"

fairness_config = {
    "S": sensitive_feature,
    "Y": target_feature,
    "S_under": sensitive_under,
    "Y_desire": "0",
}

In [39]:
# verbose = 2
# synthetic_data_count = 5000
# num_epochs = 200
# batch_size = 120

In [40]:
verbose = 2
synthetic_data_count = 100
num_epochs = 10
batch_size = 100

test_dataset = dataset.sample(n=500)

## Run models

In [41]:
# run_CTGAN(dataset, num_epochs, batch_size, verbose, synthetic_data_count)

# run_TVAE(dataset, num_epochs, batch_size, verbose, synthetic_data_count)

# run_FairGAN(dataset, num_epochs, synthetic_data_count,
#             sensitive_feature, target_feature)


# run_TabFairGAN(dataset, num_epochs, batch_size,
#                synthetic_data_count, fairness_config)

# print("DONE")

In [42]:
run_CTGAN(test_dataset, num_epochs, batch_size, verbose, synthetic_data_count)

# run_TVAE(test_dataset, num_epochs, batch_size, verbose, synthetic_data_count)

# run_FairGAN(test_dataset, num_epochs, synthetic_data_count,
#             sensitive_feature, target_feature)


run_TabFairGAN(test_dataset, num_epochs, batch_size,
               synthetic_data_count, fairness_config)

print("DONE")

[INFO]: 

Starting to train CTGAN model
[INFO]: 

Starting to train CTGAN model
[INFO]: Guidance: There are no missing values in column age. Extra column not created.
[INFO]: Guidance: There are no missing values in column age. Extra column not created.
[INFO]: Guidance: There are no missing values in column juv_fel_count. Extra column not created.
[INFO]: Guidance: There are no missing values in column juv_fel_count. Extra column not created.
[INFO]: Guidance: There are no missing values in column juv_misd_count. Extra column not created.
[INFO]: Guidance: There are no missing values in column juv_misd_count. Extra column not created.
[INFO]: Guidance: There are no missing values in column juv_other_count. Extra column not created.
[INFO]: Guidance: There are no missing values in column juv_other_count. Extra column not created.
[INFO]: Guidance: There are no missing values in column priors_count. Extra column not created.
[INFO]: Guidance: There are no missing values in column priors

Epoch 0 | D1: 3.2434 | D2: 9.4842 | G: 0.6565 | Fair: 0.6947
Epoch 0 | D1: 0.6513 | D2: 17.8394 | G: 0.7497 | Fair: 0.6934


[INFO]: Completed FairGAN_lambda_0.1
[INFO]: Completed FairGAN_lambda_0.1
[INFO]: 
Training FairGAN with λ_fair = 0.5
[INFO]: 
Training FairGAN with λ_fair = 0.5


Epoch 0 | D1: 35.5159 | D2: 18.0234 | G: 0.7527 | Fair: 0.6950


[INFO]: Completed FairGAN_lambda_0.5
[INFO]: Completed FairGAN_lambda_0.5
[INFO]: 
Training FairGAN with λ_fair = 0.7
[INFO]: 
Training FairGAN with λ_fair = 0.7


...

[INFO]: Completed TabFairGAN_lambda_0.7
[INFO]: Completed TabFairGAN_lambda_0.7
[INFO]: 
Training TabFairGAN with λ = 1.0
[INFO]: 
Training TabFairGAN with λ = 1.0
/Users/staceykagio/fair-synthetic-data-gen/venv/lib/python3.12/site-packages/sklearn/preprocessing/_data.py:2885: UserWarning: n_quantiles (2000) is greater than the total number of samples (500). n_quantiles is set to n_samples.
  warnings.warn(


Training Progress:   0%|                                                     | 0/10 [00:00<?, ?it/s]

epoch 1
epoch 2
epoch 3
epoch 4
epoch 5
epoch 6
epoch 7
epoch 8
epoch 9
epoch 10


[INFO]: Completed TabFairGAN_lambda_1.0
[INFO]: Completed TabFairGAN_lambda_1.0


DONE
